In [1]:
pip install timm

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
FINAL INFERENCE SCRIPT: DINOv3 + SSM (OFFLINE)
----------------------------------------------
- Architecture: DINOv3 (ViT-Large) + 4 SSM Layers
- Input: 448px images
- Model Source: Auto-detects 'config.json' and 'best_fold*.pth'
- Output: submission.csv (Long format)
"""

import os
import glob
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import math
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModel, AutoConfig

# ==================== CONFIGURATION ====================
class CONFIG:
    # 1. INPUT PATHS
    INPUT_ROOT = '/kaggle/input'  # Search everything here
    TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
    TEST_IMAGES = '/kaggle/input/csiro-biomass/test'
    
    # 2. MODEL PARAMETERS (Must match training)
    IMG_SIZE = 448
    NUM_SSM_LAYERS = 4
    DROPOUT = 0.1
    FEATURE_DIM = 1024
    
    # 3. INFERENCE SETTINGS
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_TTA = True  # Horizontal Flip

# ==================== HELPER: FIND BEST MODELS ====================
def find_offline_files():
    print(f"🔍 Searching {CONFIG.INPUT_ROOT}...")
    
    # 1. Find Config (The Key to Offline Mode)
    config_files = glob.glob(f'{CONFIG.INPUT_ROOT}/**/config.json', recursive=True)
    if not config_files:
        raise FileNotFoundError("❌ CRITICAL: 'config.json' not found! Did you add your training output?")
    config_path = os.path.dirname(config_files[0]) # Get the folder containing config.json
    print(f"✅ Found Config Folder: {config_path}")
    
    # 2. Find ONLY 'best_fold' Models
    all_pth = glob.glob(f'{CONFIG.INPUT_ROOT}/**/*.pth', recursive=True)
    
    # FILTER: strict check for 'best_fold'
    best_models = [p for p in all_pth if 'best_fold' in os.path.basename(p)]
    
    if not best_models:
        raise FileNotFoundError("❌ No 'best_fold' models found! Check your dataset.")
        
    print(f"✅ Found {len(best_models)} Best Models: {[os.path.basename(p) for p in best_models]}")
    
    return config_path, best_models

# Run detection immediately
try:
    CFG_FOLDER, MODEL_PATHS = find_offline_files()
except Exception as e:
    print(e)
    CFG_FOLDER = ""
    MODEL_PATHS = []

# ==================== 1. ARCHITECTURE ====================

class SimpleSSM(nn.Module):
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        x, gate = self.in_proj(x).chunk(2, dim=-1)
        x = x.transpose(1, 2)
        x = self.conv1d(x)[..., :x.shape[-1]].transpose(1, 2)
        x = F.silu(x)
        delta = torch.sigmoid(self.dt_proj(x))
        x = x * delta * torch.sigmoid(self.gate(gate))
        return self.out_proj(x)

class SSMBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.ssm = SimpleSSM(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        x = x + self.ssm(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DINOv3BiomassModel(nn.Module):
    def __init__(self, config_folder):
        super().__init__()
        
        # --- OFFLINE LOADING ---
        # 1. Load Config from local folder
        config = AutoConfig.from_pretrained(config_folder)
        
        # 2. Build Model from Config (No Internet)
        self.backbone = AutoModel.from_config(config)
        
        self.feature_dim = CONFIG.FEATURE_DIM

        # 3. Positional Encoding
        max_len = 5000
        pe = torch.zeros(max_len, self.feature_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.feature_dim, 2).float() * (-math.log(10000.0) / self.feature_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

        # 4. SSM Layers
        self.ssm_layers = nn.ModuleList([
            SSMBlock(self.feature_dim, CONFIG.DROPOUT) 
            for _ in range(CONFIG.NUM_SSM_LAYERS)
        ])
        self.norm = nn.LayerNorm(self.feature_dim)
        
        # 5. Heads
        hidden = 512
        def make_head():
            return nn.Sequential(
                nn.Linear(self.feature_dim, hidden),
                nn.ReLU(),
                nn.Dropout(CONFIG.DROPOUT),
                nn.Linear(hidden, 1),
                nn.Softplus()
            )
        
        self.head_green = make_head()
        self.head_dead = make_head()
        self.head_clover = make_head()
        
    def forward(self, x):
        # Resize
        if x.shape[2:] != (CONFIG.IMG_SIZE, CONFIG.IMG_SIZE):
            x = F.interpolate(x, size=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE), mode='bilinear')
        
        # Backbone
        outputs = self.backbone(pixel_values=x)
        features = outputs.last_hidden_state
        
        # Remove Tokens ([:, 1:, :])
        features = features[:, 1:, :] 
        
        # Add PE
        if hasattr(self, 'pe'):
             features = features + self.pe[:, :features.size(1), :]
        
        # SSM
        for layer in self.ssm_layers:
            features = layer(features)
            
        # Head
        features = self.norm(features)
        feat = features.mean(dim=1)
        
        green = self.head_green(feat)
        dead = self.head_dead(feat)
        clover = self.head_clover(feat)
        gdm = green + clover
        total = green + dead + clover
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

# ==================== 2. DATASET ====================

class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_path = str(row['image_path'])
        
        # Clean path
        clean_path = raw_path.replace('test/', '') 
        full_path = os.path.join(self.img_dir, clean_path)
        
        # Load Image
        img = cv2.imread(full_path)
        if img is None: img = cv2.imread(full_path.replace('.jpg', '.png'))
        if img is None: img = np.zeros((CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), dtype=np.uint8)
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        if self.transform:
            img = self.transform(image=img)['image']
            
        return img

# ==================== 3. INFERENCE LOOP ====================

def run_inference():
    print(f"{'='*50}")
    print(f"OFFLINE INFERENCE: DINOv3 + SSM")
    print(f"{'='*50}")
    
    if not CFG_FOLDER or not MODEL_PATHS:
        print("❌ Prerequisites missing. Check dataset.")
        return

    # 1. Prepare Data
    if not os.path.exists(CONFIG.TEST_CSV):
        print("❌ Test CSV missing.")
        return
        
    test_df = pd.read_csv(CONFIG.TEST_CSV)
    unique_test_df = test_df.drop_duplicates(subset=['image_path']).reset_index(drop=True)
    print(f"Predicting on {len(unique_test_df)} unique images.")
    
    transforms = A.Compose([
        A.Resize(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    
    dataset = TestDataset(unique_test_df, CONFIG.TEST_IMAGES, transform=transforms)
    # num_workers=0 is safer for offline kaggle kernels
    loader = DataLoader(dataset, batch_size=CONFIG.BATCH_SIZE, shuffle=False, num_workers=0) 
    
    # 2. Prediction Loop
    # We store predictions in a dictionary: {image_path: [green, dead, clover, gdm, total]}
    ensemble_sums = {path: np.zeros(5) for path in unique_test_df['image_path']}
    
    for model_path in MODEL_PATHS:
        print(f"\n🚀 Loading: {os.path.basename(model_path)}")
        
        try:
            # Initialize with Config (OFFLINE SAFE)
            model = DINOv3BiomassModel(config_folder=CFG_FOLDER).to(CONFIG.DEVICE)
            
            # Load Weights
            checkpoint = torch.load(model_path, map_location=CONFIG.DEVICE)
            
            # Handle state dict wrapper
            if 'model' in checkpoint: state_dict = checkpoint['model']
            elif 'model_state_dict' in checkpoint: state_dict = checkpoint['model_state_dict']
            else: state_dict = checkpoint
            
            # Load
            model.load_state_dict(state_dict, strict=True)
            model.eval()
            print("   ✅ Weights Loaded")
            
        except Exception as e:
            print(f"   ⚠️ FAILED to load: {e}")
            continue
            
        fold_preds = []
        with torch.no_grad():
            for imgs in tqdm(loader, leave=False):
                imgs = imgs.to(CONFIG.DEVICE)
                
                # Forward
                preds_1 = model(imgs)
                
                if CONFIG.USE_TTA:
                    imgs_flip = torch.flip(imgs, dims=[3])
                    preds_2 = model(imgs_flip)
                    final_batch_preds = (preds_1 + preds_2) / 2.0
                else:
                    final_batch_preds = preds_1
                
                fold_preds.append(final_batch_preds.cpu().numpy())
        
        fold_preds = np.concatenate(fold_preds, axis=0)
        
        # Add to ensemble
        for i, path in enumerate(unique_test_df['image_path']):
            ensemble_sums[path] += fold_preds[i]
            
        del model
        torch.cuda.empty_cache()

    # 3. Create Submission (Strict Format Check)
    print("\nGenerating submission.csv...")
    final_targets = []
    num_models = len(MODEL_PATHS)
    
    # Iterate through the original test_df to preserve order and structure
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        path = row['image_path']
        t_name = row['target_name']
        
        # Get average vector for this image [Green, Dead, Clover, GDM, Total]
        avg_vec = ensemble_sums.get(path, np.zeros(5)) / num_models
        
        # Map specific target column to value
        val = 0.0
        if t_name == 'Dry_Green_g':   val = avg_vec[0]
        elif t_name == 'Dry_Dead_g':  val = avg_vec[1]
        elif t_name == 'Dry_Clover_g': val = avg_vec[2]
        elif t_name == 'GDM_g':       val = avg_vec[3]
        elif t_name == 'Dry_Total_g': val = avg_vec[4]
        
        # Biomass cannot be negative
        final_targets.append(max(0.0, float(val)))
        
    sub = pd.DataFrame({'sample_id': test_df['sample_id'], 'target': final_targets})
    sub.to_csv('submission.csv', index=False)
    
    print("\n✅ submission.csv Saved!")
    print("Format Verification:")
    print(sub.head(10)) # Print 10 rows to verify the grouping

if __name__ == '__main__':
    run_inference()

/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


🔍 Searching /kaggle/input...
✅ Found Config Folder: /kaggle/input/dinov3-ssm/dinov2_checkpoints/dinov3_config
✅ Found 5 Best Models: ['best_fold2.pth', 'best_fold4.pth', 'best_fold1.pth', 'best_fold3.pth', 'best_fold0.pth']
OFFLINE INFERENCE: DINOv3 + SSM
Predicting on 1 unique images.

🚀 Loading: best_fold2.pth


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

   ✅ Weights Loaded


  0%|          | 0/1 [00:00<?, ?it/s]


🚀 Loading: best_fold4.pth
   ✅ Weights Loaded


... (6 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]


Generating submission.csv...


  0%|          | 0/5 [00:00<?, ?it/s]


✅ submission.csv Saved!
Format Verification:
                    sample_id     target
0  ID1001187975__Dry_Clover_g   7.351000
1    ID1001187975__Dry_Dead_g  13.946862
2   ID1001187975__Dry_Green_g  35.579932
3   ID1001187975__Dry_Total_g  56.877792
4         ID1001187975__GDM_g  42.930931
